<a href="https://colab.research.google.com/github/anu-Prakash-19/AI_Driven_Customer_Analytics_for_Revenue_Prediction_and_Churn_Analysis/blob/main/AI_MODEL_GENCAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%cd /content
!rm -rf graphcast
!git clone https://github.com/google-deepmind/graphcast.git
%cd /content/graphcast
!git log --oneline -- graphcast/rollout.py | head -n 3
!echo "--- top level now:"
!ls

In [ ]:
%pip install -U importlib_metadata

In [ ]:
%pip install --upgrade https://github.com/deepmind/graphcast/archive/master.zip

Imports

In [ ]:
!pip uninstall -y shapely

In [ ]:
!pip install shapely --no-binary shapely

In [ ]:
!pip install -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

In [ ]:
!pip show dm-haiku

In [ ]:
%pip install dm-haiku

In [ ]:
%pip show dm-haiku

In [ ]:
%pip install --upgrade https://github.com/google-deepmind/graphcast/archive/master.zip

In [ ]:
%pip show graphcast

In [ ]:
%pip install --upgrade https://github.com/google-deepmind/graphcast/archive/master.zip 2>&1 | tail -n 30

In [ ]:
!curl -sI https://github.com | head -n 1

In [ ]:
!git clone https://github.com/google-deepmind/graphcast.git
import sys, os
sys.path.insert(0, os.path.abspath("graphcast"))
%pip install chex jraph dm-haiku xarray-tensorstore dask netCDF4 scipy
import graphcast
print(graphcast.__file__)

In [ ]:
import sys, os, importlib

# 1. Clear the stale namespace-package entry
for m in [m for m in sys.modules if m == "graphcast" or m.startswith("graphcast.")]:
    del sys.modules[m]

# 2. Find where the real package is (the folder containing __init__.py)
!find / -type d -name graphcast -path "*graphcast/graphcast" 2>/dev/null | head

In [ ]:
import os
print("cwd:", os.getcwd())
!ls -la
!find / -name "rollout.py" -path "*graphcast*" 2>/dev/null

In [ ]:
%pip install git+https://github.com/google-deepmind/weathernext.git@v0.3.0


In [ ]:
!find /content/graphcast -name "gencast*.py" -o -name "denoiser*.py" -o -name "nan_cleaning*.py" -o -name "xarray_jax*.py" -o -name "normalization*.py" -o -name "checkpoint*.py" -o -name "data_utils*.py" -o -name "xarray_tree*.py"

In [ ]:
%cd /content
!rm -rf graphcast
!git clone https://github.com/google-deepmind/graphcast.git
%cd /content/graphcast
!git log --oneline -- graphcast/rollout.py | head -n 3

In [ ]:
  !git checkout <hash>^
  !ls graphcast | head

In [ ]:
  %pip install -e /content/graphcast
  %cd /content

In [ ]:
!git checkout abc1234^        # replace abc1234 with the hash from Step 2
!ls graphcast | head          # you should now see rollout.py, gencast.py, etc.
%pip install -e /content/graphcast
%cd /content

In [ ]:
!git checkout <hash>^
!ls graphcast | head

In [ ]:
%pip install -e /content/graphcast
%cd /content

In [ ]:
%cd /content
!rm -rf graphcast
!git clone https://github.com/google-deepmind/graphcast.git
%cd /content/graphcast
!git log --oneline -- graphcast/rollout.py | head -n 3
!ls

In [ ]:
%cd /content
!rm -rf graphcast gc_repo
!git clone https://github.com/google-deepmind/graphcast.git gc_repo
%cd /content/gc_repo
!h=$(git log -1 --format=%H -- graphcast/rollout.py); echo "last commit touching rollout.py: $h"; git checkout ${h}^ 2>&1 | tail -n 2
!ls graphcast | head -n 20
%pip install -e /content/gc_repo
%cd /content

In [ ]:
!find /content/gc_repo -name "gencast*.py" -o -name "denoiser*.py" -o -name "nan_cleaning*.py" -o -name "xarray_jax*.py" -o -name "normalization*.py" -o -name "checkpoint*.py" -o -name "data_utils*.py" -o -name "xarray_tree*.py"

In [ ]:
!pip install --upgrade https://github.com/deepmind/graphcast/archive/master.zip

In [ ]:
!ls /content/gc_repo

In [ ]:
import sys
sys.path.insert(0, "/content/gc_repo")
import graphcast
print(graphcast.__file__)

In [ ]:
import sys
sys.path.insert(0, "/content/gc_repo")

In [ ]:
!pip install dinosaur-dycore xarray_tensorstore

In [ ]:
!pip show graphcast
import graphcast
print(graphcast.__file__)

In [ ]:
#title imports
import sys
import site
import importlib

# Force refresh site-packages to ensure newly installed packages are visible
for path in site.getsitepackages() + [site.getusersitepackages()]:
    if path not in sys.path:
        sys.path.insert(0, path)

importlib.invalidate_caches()

import dataclasses
import datetime
import math
from google.cloud import storage
from typing import Optional
import haiku as hk
from IPython.display import HTML
from IPython import display
import ipywidgets as widgets
import jax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray

from graphcast import rollout
from graphcast import xarray_jax
from graphcast import normalization
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import xarray_tree
from graphcast import gencast
from graphcast import denoiser
from graphcast import nan_cleaning

Plotting function

In [ ]:
!ls /content/gc_repo/graphcast | head -n 5
%pip show graphcast | head -n 3

In [ ]:
import datetime
import math
from typing import Optional

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray
from IPython.display import HTML


def select(
    data: xarray.Dataset,
    variable: str,
    level: Optional[int] = None,
    max_steps: Optional[int] = None,
) -> xarray.Dataset:
    data = data[variable]
    if "batch" in data.dims:
        data = data.isel(batch=0)
    if max_steps is not None and "time" in data.sizes and max_steps < data.sizes["time"]:
        data = data.isel(time=range(0, max_steps))
    if level is not None and "level" in data.coords:
        data = data.sel(level=level)
    return data


def scale(
    data: xarray.Dataset,
    center: Optional[float] = None,
    robust: bool = False,
) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:
    vmin = np.nanpercentile(data, (2 if robust else 0))
    vmax = np.nanpercentile(data, (98 if robust else 100))
    if center is not None:
        diff = max(vmax - center, center - vmin)
        vmin = center - diff
        vmax = center + diff
    return (data, matplotlib.colors.Normalize(vmin, vmax),
            ("RdBu_r" if center is not None else "viridis"))


def plot_data(
    data: dict,
    fig_title: str,
    plot_size: float = 5,
    robust: bool = False,
    cols: int = 4,
):
    first_data = next(iter(data.values()))[0]
    max_steps = first_data.sizes.get("time", 1)
    assert all(max_steps == d.sizes.get("time", 1) for d, _, _ in data.values())

    cols = min(cols, len(data))
    rows = math.ceil(len(data) / cols)
    figure = plt.figure(figsize=(plot_size * 2 * cols, plot_size * rows))
    figure.suptitle(fig_title, fontsize=16)
    figure.subplots_adjust(wspace=0, hspace=0)
    figure.tight_layout()

    images = []
    for i, (title, (plot_d, norm, cmap)) in enumerate(data.items()):
        ax = figure.add_subplot(rows, cols, i + 1)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(title)
        im = ax.imshow(
            plot_d.isel(time=0, missing_dims="ignore"),
            norm=norm, origin="lower", cmap=cmap)
        plt.colorbar(
            mappable=im, ax=ax, orientation="vertical",
            pad=0.02, aspect=16, shrink=0.75, cmap=cmap,
            extend=("both" if robust else "neither"))
        images.append(im)

    def update(frame):
        if "time" in first_data.dims:
            td = datetime.timedelta(
                microseconds=first_data["time"][frame].item() / 1000)
            figure.suptitle(f"{fig_title}, {td}", fontsize=16)
        else:
            figure.suptitle(fig_title, fontsize=16)
        for im, (plot_d, norm, cmap) in zip(images, data.values()):
            im.set_data(plot_d.isel(time=frame, missing_dims="ignore"))

    ani = animation.FuncAnimation(
        fig=figure, func=update, frames=max_steps, interval=250)
    plt.close(figure.number)
    return HTML(ani.to_jshtml())

In [ ]:
from google.cloud import storage

gcs_client = storage.Client.create_anonymous_client()
gcs_bucket = gcs_client.get_bucket("dm_graphcast")
dir_prefix = "gencast/"

Choose The Model

In [ ]:
# @title Choose the model

params_file_options = [
    name for blob in gcs_bucket.list_blobs(prefix=(dir_prefix+"params/"))
    if (name := blob.name.removeprefix(dir_prefix+"params/"))]  # Drop empty string.

latent_value_options = [int(2**i) for i in range(4, 10)]
random_latent_size = widgets.Dropdown(
    options=latent_value_options, value=512, description="Latent size:")
random_attention_type = widgets.Dropdown(
    options=["splash_mha", "triblockdiag_mha", "mha"], value="splash_mha", description="Attention:")
random_mesh_size = widgets.IntSlider(
    value=4, min=4, max=6, description="Mesh size:")
random_num_heads = widgets.Dropdown(
    options=[int(2**i) for i in range(0, 3)], value=4, description="Num heads:")
random_attention_k_hop = widgets.Dropdown(
    options=[int(2**i) for i in range(2, 5)], value=16, description="Attn k hop:")

def update_latent_options(*args):
  def _latent_valid_for_attn(attn, latent, heads):
    head_dim, rem = divmod(latent, heads)
    if rem != 0:
      return False
    # Required for splash attn.
    if head_dim % 128 != 0:
      return attn != "splash_mha"
    return True
  attn = random_attention_type.value
  heads = random_num_heads.value
  random_latent_size.options = [
      latent for latent in latent_value_options
      if _latent_valid_for_attn(attn, latent, heads)]

# Observe changes to only allow for valid combinations.
random_attention_type.observe(update_latent_options, "value")
random_latent_size.observe(update_latent_options, "value")
random_num_heads.observe(update_latent_options, "value")

params_file = widgets.Dropdown(
    options=[f for f in params_file_options if "Mini" in f],
    description="Params file:",
    layout={"width": "max-content"})

source_tab = widgets.Tab([
    widgets.VBox([
        random_attention_type,
        random_mesh_size,
        random_num_heads,
        random_latent_size,
        random_attention_k_hop
    ]),
    params_file,
])
source_tab.set_title(0, "Random")
source_tab.set_title(1, "Checkpoint")
widgets.VBox([
    source_tab,
    widgets.Label(value="Run the next cell to load the model. Rerunning this cell clears your selection.")
])

In [ ]:
# @title Load the model

source = source_tab.get_title(source_tab.selected_index)

if source == "Random":
  params = None  # Filled in below.
  state = {}
  task_config = gencast.TASK
  # Use default values.
  sampler_config = gencast.SamplerConfig()
  noise_config = gencast.NoiseConfig()
  noise_encoder_config = denoiser.NoiseEncoderConfig()
  # Configure, otherwise use default values.
  denoiser_architecture_config = denoiser.DenoiserArchitectureConfig(
      sparse_transformer_config = denoiser.SparseTransformerConfig(
          attention_k_hop=random_attention_k_hop.value,
          attention_type=random_attention_type.value,
          d_model=random_latent_size.value,
          num_heads=random_num_heads.value
          ),
      mesh_size=random_mesh_size.value,
      latent_size=random_latent_size.value,
  )
else:
  assert source == "Checkpoint"
  with gcs_bucket.blob(dir_prefix + f"params/{params_file.value}").open("rb") as f:
    ckpt = checkpoint.load(f, gencast.CheckPoint)
  params = ckpt.params

In [ ]:
# @title Get and filter the list of available example datasets

dataset_file_options = [
    name for blob in gcs_bucket.list_blobs(prefix=(dir_prefix + "dataset/"))
    if (name := blob.name.removeprefix(dir_prefix+"dataset/"))]  # Drop empty string.

def parse_file_parts(file_name):
  return dict(part.split("-", 1) for part in file_name.split("_"))


def data_valid_for_model(file_name: str, params_file_name: str):
  """Check data type and resolution matches."""
  if source == "Random":
    return True
  data_file_parts = parse_file_parts(file_name.removesuffix(".nc"))
  res_matches = data_file_parts["res"].replace(".", "p") in params_file_name.lower()
  source_matches = "Operational" in params_file_name
  if data_file_parts["source"] == "era5":
    source_matches = not source_matches
  return res_matches and source_matches

dataset_file = widgets.Dropdown(
    options=[
        (", ".join([f"{k}: {v}" for k, v in parse_file_parts(option.removesuffix(".nc")).items()]), option)
        for option in dataset_file_options
        if data_valid_for_model(option, params_file.value)
    ],
    description="Dataset file:",
    layout={"width": "max-content"})
widgets.VBox([
    dataset_file,
    widgets.Label(value="Run the next cell to load the dataset. Rerunning this cell clears your selection and refilters the datasets that match ...")  # text cut off in screenshot
])

In [ ]:
# @title Load weather data

with gcs_bucket.blob(dir_prefix+f"dataset/{dataset_file.value}").open("rb") as f:
  example_batch = xarray.load_dataset(f).compute()

assert example_batch.dims["time"] >= 3  # 2 for input, >=1 for targets

print(", ".join([f"{k}: {v}" for k, v in parse_file_parts(dataset_file.value.removesuffix(".nc")).items()]))

example_batch

In [ ]:
# @title Choose data to plot

plot_example_variable = widgets.Dropdown(
    options=example_batch.data_vars.keys(),
    value="2m_temperature",
    description="Variable")
plot_example_level = widgets.Dropdown(
    options=example_batch.coords["level"].values,
    value=500,
    description="Level")
plot_example_robust = widgets.Checkbox(value=True, description="Robust")
plot_example_max_steps = widgets.IntSlider(
    min=1, max=example_batch.dims["time"], value=example_batch.dims["time"],
    description="Max steps")

widgets.VBox([
    plot_example_variable,
    plot_example_level,
    plot_example_robust,
    plot_example_max_steps,
    widgets.Label(value="Run the next cell to plot the data. Rerunning this cell clears your selection.")
])

In [ ]:
# @title Plot example data

plot_size = 7

data = {
    " ": scale(select(example_batch, plot_example_variable.value, plot_example_level.value, plot_example_max_steps.value),
               robust=plot_example_robust.value),
}
fig_title = plot_example_variable.value
if "level" in example_batch[plot_example_variable.value].coords:
  fig_title += f" at {plot_example_level.value} hPa"

plot_data(data, fig_title, plot_size, plot_example_robust.value)